# Problem 2: Kalman Filter for Landmark Localization
All of your code goes in `P2_kf.py`. The robot pose is known; the filter estimates the positions of the four landmarks from noisy measurements of their positions in the world frame.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt

from P2_kf import landmark_model, kf_predict, kf_update, run_kf
from utils import plot_error_ellipse

os.makedirs("plots", exist_ok=True)
data = np.load("data/filter_data.npz")
time, landmarks, path = data["time"], data["landmarks"], data["kf_path"]
zs, mu0, Sigma0, Q = data["kf_z"], data["kf_mu0"], data["kf_Sigma0"], data["kf_Q"]
print("time steps:", len(zs), " measurement noise Q = %.2f I" % Q[0, 0])

## Running the filter

In [ ]:
A, C, R = landmark_model()
mus, Sigmas = run_kf(mu0, Sigma0, zs, A, C, R, Q)

truth = landmarks.ravel()
err = np.linalg.norm((mus - truth).reshape(len(mus), 4, 2), axis=2)
print("initial position error of each landmark (m):", err[0].round(3))
print("final position error of each landmark (m):  ", err[-1].round(3))
print("final standard deviation of each coordinate (m):", np.sqrt(np.diag(Sigmas[-1])).round(4))

## Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14.5, 4.5))
for k, name in enumerate(["x", "y"]):
    for j in range(4):
        axes[k].plot(time, np.full(len(time), landmarks[j, k]), color=f"C{j}", lw=2, label=f"m{j + 1} true")
        axes[k].plot(time, mus[:, 2 * j + k], ".", color=f"C{j}", ms=3, label=f"m{j + 1} estimate")
    axes[k].set_xlabel("time (s)")
    axes[k].set_ylabel(f"landmark {name} (m)")
    axes[k].set_title(f"Kalman filter estimates of the landmark {name} coordinates")
    axes[k].grid(True)
axes[1].legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.02, 1.0))
plt.tight_layout()
plt.savefig("plots/p2_kf_time.png", dpi=120)
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(path[:, 0], path[:, 1], lw=2, label="robot")
ax.plot(landmarks[:, 0], landmarks[:, 1], "ks", ms=8, label="landmarks")
for j in range(4):
    ax.plot(mus[:, 2 * j], mus[:, 2 * j + 1], "+", color="orange", ms=6, label="estimates" if j == 0 else None)
    for t in [0, 2, 5, 10, len(mus) - 1]:
        plot_error_ellipse(ax, mus[t, 2 * j:2 * j + 2], Sigmas[t][2 * j:2 * j + 2, 2 * j:2 * j + 2])
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("Landmark estimates and 95% ellipses at t = 0, 0.2, 0.5, 1 s and the end")
ax.axis("equal")
ax.grid(True)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("plots/p2_kf_map.png", dpi=120)
plt.show()